In [5]:
import random
import numpy as np
import math 
from random import choice
import statistics 

In [440]:
def solveSudoku(sudoku, decreaseFactor=0.99):
    """
    Run puzzle solve algorithm.
    """
    sigma = CalculateInitialSigma(sudoku).item()
    fixed_cells = np.minimum(sudoku, 1)
    n_iter = np.sum(1-fixed_cells).item()

    sudoku = RandomlyFill3x3Blocks(np.copy(sudoku))
    score = CalculateNumberOfErrors(sudoku)
    stuckCount = 0
    if score:
        while True:
            previousScore = score
            for iter in range(n_iter):
                sudoku, scoreDiff = ChooseNewState(sudoku, fixed_cells, sigma)
                score += scoreDiff
                if score <= 0:  
                    break
            assert CalculateNumberOfErrors(sudoku)==score
            
            print(score, sigma)
            
            if score <= 0:
                break
            
            sigma *= decreaseFactor
            if score >= previousScore:
                stuckCount += 1
            else:
                stuckCount = 0
            if stuckCount > 80:
                sigma += 2
    return sudoku


def CalculateInitialSigma(sudoku, sample_size=200):
    """
    Calculate initial temperature parameter as stddev of cost of N random starting states
    """
    return np.std([CalculateNumberOfErrors(RandomlyFill3x3Blocks(sudoku)) 
                   for i in range(200)])
    

def CalculateNumberOfErrors(sudoku):
    """
    Calculate duplicate numbers in grid rows and columns
    """
    return np.sum([np.sum(np.maximum(np.sum(sudoku == j, axis=i)-1,0)) 
                   for j in range(1,10) for i in [0,1]]).item()
    

def RandomlyFill3x3Blocks(sudoku):
    """
    Randomly populate unfilled boxes such that each 3*3 block has digits 1-9
    """
    sudoku = np.copy(sudoku)
    for block in range(9):
        boxes = sudoku[(y:=3*(block//3)):y+3, (x:=3*(block%3)):x+3].reshape(-1)
        boxes[np.arange(9)[boxes==0]] = np.random.permutation([*{*range(1,10)}-{*boxes}])
        sudoku[y:y+3, x:x+3] = boxes.reshape(3,3)
    return sudoku


def ChooseNewState(sudoku, fixed_cells, sigma):
    """
    Make a random switch between cells in one block, choose whether to accept, return result.
    """
    poss = np.arange(81)[fixed_cells.reshape(-1) == 0]
    while True:
        c1 = np.random.choice(poss).item()
        if (c2p:=poss[(poss//9//3==c1//9//3)&(poss%9//3==c1%9//3)&(poss!=c1)]).shape[0]:
            c2 = np.random.choice(c2p).item()
            break
    sudoku_new = (sudoku.reshape(-1)
                [np.vectorize(lambda x:{c1:c2,c2:c1}.get(x,x))
                (np.arange(81))].reshape(9,9))
    currentCost, newCost = [36-sum([np.unique([s[v],s[:,v]][n]).shape[0]
                                    for c in [c1,c2]
                                    for n,v in enumerate([c//9,c%9])])
                            for s in [sudoku,sudoku_new]]
    costDifference = newCost - currentCost
    rho = math.exp(-costDifference / sigma)

    return [[sudoku,0], [sudoku_new,costDifference]][np.random.random() < rho]


def PrintSudoku(sudoku):
    """
    Pretty(ish) print sudoku grid
    """
    for line in [r
                 for i in range(9)
                 for r in [' | '.join([' '.join([str(k) for k in j]) 
                                       for j in sudoku[i].reshape(3,3).tolist()]),
                           *[[],['-'*21]][i in[2,5]]]]:
        print(line)

In [418]:
# Original puzzle
# startingSudoku = """
#                     024007000
#                     600000000
#                     003680415
#                     431005000
#                     500000032
#                     790000060
#                     209710800
#                     040093000
#                     310004750
#                 """
                
# Diabolical puzzle
startingSudoku = """
                    170008000
                    006010004
                    004300000
                    700900080
                    002701600
                    060005007
                    000004900
                    800020400
                    000100035
                """

sudoku = np.array([[int(i) for i in line] for line in startingSudoku.split()])
PrintSudoku(solveSudoku(sudoku))

43 4.587785413464758
40 4.54190755933011
39 4.496488483736809
39 4.451523598899441
40 4.407008362910447
42 4.362938279281343
36 4.319308896488529
43 4.276115807523643
44 4.233354649448407
42 4.191021102953923
40 4.1491108919243835
41 4.1076197830051395
34 4.066543585175088
41 4.025878149323337
36 3.985619367830104
33 3.9457631741518027
41 3.906305542410285
41 3.867242486986182
44 3.82857006211632
35 3.7902843614951567
40 3.752381517880205
33 3.714857702701403
33 3.6777091256743892
40 3.6409320344176455
47 3.604522714073469
33 3.5684774869327343
38 3.532792712063407
37 3.497464784942773
46 3.462490137093345
42 3.4278652357224115
36 3.3935865833651873
39 3.3596507175315353
42 3.32605421035622
44 3.2927936682526577
40 3.259865731570131
43 3.2272670742544296
41 3.194994403511885
32 3.163044459476766
32 3.1314140148819987
36 3.1000998747331785
37 3.069098875985847
43 3.0384078872259885
33 3.0080238083537285
42 2.9779435702701913
49 2.9481641345674894
42 2.9186824932218145
45 2.8894956682895

In [443]:
# simulated annealing search of a one-dimensional objective function
from numpy import asarray
from numpy import exp
from numpy.random import randn
from numpy.random import rand
from numpy.random import seed

# objective function
def objective(x):
	return x[0]**4 - 5*x[0]**2 - 29*x[0] + 906

# simulated annealing algorithm
def simulated_annealing(objective, bounds, n_iterations, step_size, temp):
	# generate an initial point
	best = bounds[:, 0] + rand(len(bounds)) * (bounds[:, 1] - bounds[:, 0])
	# evaluate the initial point
	best_eval = objective(best)
	# current working solution
	curr, curr_eval = best, best_eval
	# run the algorithm
	for i in range(n_iterations):
		# take a step
		candidate = curr + randn(len(bounds)) * step_size
		# evaluate candidate point
		candidate_eval = objective(candidate)
		# check for new best solution
		if candidate_eval < best_eval:
			# store new best point
			best, best_eval = candidate, candidate_eval
			# report progress
			print('>%d f(%s) = %.5f' % (i, best, best_eval))
		# difference between candidate and current point evaluation
		diff = candidate_eval - curr_eval
		# calculate temperature for current epoch
		t = temp / float(i + 1)
		# calculate metropolis acceptance criterion
		metropolis = exp(-diff / t)
		# check if we should keep the new point
		if diff < 0 or rand() < metropolis:
			# store the new current point
			curr, curr_eval = candidate, candidate_eval
	return [best, best_eval]

# seed the pseudorandom number generator
seed(1)
# define range for input
bounds = asarray([[-5.0, 5.0]])
# define the total iterations
n_iterations = 1000
# define the maximum step size
step_size = 0.1
# initial temperature
temp = 10
# perform the simulated annealing search
best, score = simulated_annealing(objective, bounds, n_iterations, step_size, temp)
print('Done!')
print('f(%s) = %f' % (best, score))

>6 f([-0.74493804]) = 925.13649
>8 f([-0.68850356]) = 923.82113
>10 f([-0.57512661]) = 921.13423
>12 f([-0.52174552]) = 919.84363
>15 f([-0.41647048]) = 917.24049
>17 f([-0.32038848]) = 914.78856
>29 f([-0.16622629]) = 910.68317
>30 f([0.03797659]) = 904.89147
>31 f([0.08272866]) = 903.56670
>32 f([0.15106708]) = 901.50547
>33 f([0.15335568]) = 901.43565
>34 f([0.23907911]) = 898.78418
>35 f([0.25747216]) = 898.20624
>37 f([0.38247717]) = 894.19812
>39 f([0.44113949]) = 892.27181
>40 f([0.4999689]) = 890.31354
>41 f([0.53465484]) = 889.14744
>42 f([0.67135811]) = 884.48016
>43 f([0.73872971]) = 882.14604
>46 f([0.74675477]) = 881.86686
>47 f([0.76957865]) = 881.07172
>49 f([0.79739154]) = 880.10076
>50 f([1.01594908]) = 872.44205
>52 f([1.04845482]) = 871.30689
>54 f([1.08001831]) = 870.20785
>56 f([1.1517773]) = 867.72535
>60 f([1.17160727]) = 867.04428
>61 f([1.18350813]) = 866.63674
>63 f([1.22126451]) = 865.35043
>64 f([1.27945703]) = 863.39050
>66 f([1.39934882]) = 859.46246
>67 f